# Final model: graph stacking with a stacked MLP blend

This notebook builds the final submission model I submitted on Kaggle from end to end: 

- data checks and preprocessing
- the chronological validation checks
- graph feature engineering and two-round graph stacking (D2 LightGBM and D2 HGB)
- an MLP trained on the stacked features
- validation of every component and blend ratio
- the final `.csv` submissions.

**Final submissions.** `FINAL_RATIOS` lists the LGBM : HGB : MLP rank-blend ratios written to `submission/`: **3:1:1** and **1:1:1**.

Run the cells from top to bottom. The full run takes about 15 minutes on CPU.

In [ ]:
from pathlib import Path
import itertools
import json
import math
import subprocess
import sys
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# Please put the competition data in a folder named "bt-4012-competition-2026" in the root of this repository.
DATA_FOLDER = "bt-4012-competition-2026"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / DATA_FOLDER).is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"Open this notebook inside the project containing {DATA_FOLDER}.")
DATA = ROOT / DATA_FOLDER

FEATURES = [f"feat_{i}" for i in range(1, 166)]

WORK = ROOT / "work"              # intermediate files (stage-2 features, MLP predictions)
SUBMISSIONS = ROOT / "submission"  # final submission files inside this folder
WORK.mkdir(exist_ok=True)
SUBMISSIONS.mkdir(exist_ok=True)

FINAL_RATIOS = [(3, 1, 1), (1, 1, 1)]   # LGBM : HGB : MLP; 3:1:1 is the top submission
HALF_LIFE = 15
TOP_FEATURES = 30
N_FOLDS = 5
HGB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,
                  l2_regularization=0.0, class_weight=None, early_stopping=False, random_state=42)
LGBM_PARAMS = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=20,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.5, verbose=-1, random_state=42)
print("HGB:", HGB_PARAMS)
print("LightGBM:", LGBM_PARAMS)

HGB: {'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'class_weight': None, 'early_stopping': False, 'random_state': 42}
LightGBM: {'n_estimators': 600, 'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 20, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.5, 'verbose': -1, 'random_state': 42}


## 1. Load the data and check it
`read_csv` converts blank labels to `NaN`. **Unknown is not licit.** Unknown rows must be excluded from the supervised loss and from scoring, never filled with 0. 

Features are read as `float32` to keep the stacked matrices compact.

In [2]:
dtypes = {f: "float32" for f in FEATURES}
train = pd.read_csv(DATA / "train.csv", dtype=dtypes)
test = pd.read_csv(DATA / "test.csv", dtype=dtypes)
edges = pd.read_csv(DATA / "txs_edgelist.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

raw_labels = pd.read_csv(DATA / "train.csv", usecols=["label"], dtype="string", na_filter=False)["label"]
unexpected = sorted(set(raw_labels) - {"", "0", "1"})
assert not unexpected, f"Unexpected stored labels: {unexpected}"
display(raw_labels.value_counts().rename(index={"": "unknown (blank)", "0": "licit (0)", "1": "illicit (1)"}).to_frame("rows"))

checks = {
    "train txId unique": train.txId.is_unique,
    "test txId unique": test.txId.is_unique,
    "train and test txIds disjoint": set(train.txId).isdisjoint(test.txId),
    "sample_submission index equals test index": sample_submission["index"].equals(test["index"]),
    "train steps end before test steps begin": train.time_step.max() < test.time_step.min(),
    "no missing or infinite features": bool(np.isfinite(train[FEATURES].to_numpy()).all()
                                            and np.isfinite(test[FEATURES].to_numpy()).all()),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values())
print(f"Train rows: {len(train):,}; test rows: {len(test):,}; edges: {len(edges):,}")

,rows
label,
unknown (blank),110537
licit (0),27591
illicit (1),3644


,passed
train txId unique,True
test txId unique,True
train and test txIds disjoint,True
sample_submission index equals test index,True
train steps end before test steps begin,True
no missing or infinite features,True


Train rows: 141,772; test rows: 15,329; edges: 234,355


## Validation checks
All validation uses three chronological checks:

| Check | Fit on known labels | Score on known labels | Horizon | Role |
|---|---|---|---|---|
| `steps_22_28` | steps 1–21 | steps 22–28 | 7 steps | Main check for **comparing** models; it has the most room between them. |
| `steps_29_35` | steps 1–28 | steps 29–35 | 7 steps | Closest to the test period; scores are usually very high, so it separates models less. |
| `14_step_check` | steps 1–21 (the **same** fitted model as `steps_22_28`) | steps 22–35 | 14 steps | Same horizon as the real task: fit ≤ 35, predict 36–49. |

During my experiments, I found that 14-step check score is always lower than the 2 7-step check scores.

Why the 14-step check usually scores lower than `steps_22_28`:
- **Longer horizon.** The model must rank transactions up to 14 steps after its last training step, and the far half (29–35) has drifted further from steps 1–21.
- **No refit at step 28.** The `steps_29_35` check refits on 1–28 before scoring 29–35; the 14-step check does not.
- **Pooled AUC across steps.** Scores from 14 different steps are ranked together, so any step-to-step shift in score level lowers AUC even when the ranking within each step is good.

Rules:
- Fit and score on **known labels only**; unknown rows never count as licit.
- Choose settings using the mean ROC-AUC of the two 7-step checks (`steps_22_28` and `steps_29_35`). Report all three checks; never use the 14-step check for selection.

In [3]:
CHECKS = [
    {"name": "steps_22_28", "train_start": 1, "train_end": 21, "validation_start": 22, "validation_end": 28},
    {"name": "steps_29_35", "train_start": 1, "train_end": 28, "validation_start": 29, "validation_end": 35},
    {"name": "14_step_check", "train_start": 1, "train_end": 21, "validation_start": 22, "validation_end": 35},
]
SELECTION_CHECKS = ["steps_22_28", "steps_29_35"]  # the 14-step check is reported, never used for selection
known = train[train.label.notna()]
for check in CHECKS:
    fit = known[known.time_step.between(check["train_start"], check["train_end"])]
    valid = known[known.time_step.between(check["validation_start"], check["validation_end"])]
    assert fit.time_step.max() < valid.time_step.min(), check["name"]
    assert set(fit.txId).isdisjoint(valid.txId), check["name"]
    assert fit.label.nunique() == valid.label.nunique() == 2, check["name"]
    print(f'{check["name"]:>14}: fit {len(fit):,} rows, validate {len(valid):,} rows '
          f'({int(valid.label.sum()):,} illicit)')

   steps_22_28: fit 19,530 rows, validate 5,677 rows (671 illicit)
   steps_29_35: fit 25,207 rows, validate 6,028 rows (1,102 illicit)
 14_step_check: fit 19,530 rows, validate 11,705 rows (1,773 illicit)


## 2. Graph feature engineering and graph stacking
The model adds graph features in 5 stages:

| Stage | Adds |
|---|---|
| A | nothing: the 165 supplied features (LightGBM here only ranks features for stage C) |
| B | structure: in/out degree, in/out visible-neighbour counts and visible fractions (7 columns) |
| C | mean/max over **visible** neighbours of the top-30 features (by LightGBM gain on the fitting rows), plus own-minus-neighbour-mean for those 30 |
| D1 | graph stacking, round 1: out-of-fold model scores, then their neighbour mean/max/min/count |
| D2 | round 2: the same again with the round-1 stacked scores (mean/max), so information reaches two hops |

Two D2 models are built on stage C features: **D2 HGB** and **D2 LightGBM**.

**Visible neighbours.** A neighbour is *visible* if it is a labelled fitting row or a row the model is asked to score. Unknown-label rows and hidden edge-list IDs are never visible. This definition means the same thing when scoring validation steps and when scoring the test steps.

**Out-of-fold scores.** Stacked features for labelled fitting rows come from 5 folds grouped by **time step**, so a row's own label never shapes its own score; rows being scored get scores from a model fitted on all fitting rows.

**Recency weights.** All models weight each labelled row by `0.5 ** ((latest_fit_step - row_step) / 15)` (half-life 15 steps). So more recent rows will contribute more in terms of weights.

`neighbour_agg` aggregates values over undirected neighbours that pass a mask; edge-list IDs without a row are skipped. `structure` counts degrees from the full edge list and visible neighbours from the mask.

In [ ]:
def neighbour_agg(df, values, mask, prefix, stats=("mean", "max")):
    """Aggregate `values` over undirected neighbours where `mask` is True, aligned to df."""
    pos = pd.Series(np.arange(len(df)), index=df.txId.values) #df: the transactions (rows) being processed
    a = pos.reindex(edges.txId1.values).to_numpy(); b = pos.reindex(edges.txId2.values).to_numpy()
    ok = ~(np.isnan(a) | np.isnan(b)); a = a[ok].astype(int); b = b[ok].astype(int)
    src = np.concatenate([a, b]); dst = np.concatenate([b, a])  # dst's neighbour is src, both directions
    keep = mask[src]; src, dst = src[keep], dst[keep]
    #mask - a true/false flag per row saying whether that row counts as a neighbour
    g = pd.DataFrame(values.to_numpy()[src], columns=values.columns); g["_dst"] = dst 
    #values: the columns to summarise (e.g. the top-30 features)
    agg = g.groupby("_dst").agg(list(stats))
    #stats: which summaries to compute (here is mean and max)
    agg.columns = [f"{prefix}_{c}_{s}" for c, s in agg.columns]
    #prefix: a label added to the output column names
    out = agg.reindex(np.arange(len(df))); out.index = df.index
    return out


def structure(df):
    """Degrees from the full edge list; visible-neighbour counts and fractions."""
    vis = pd.Series(df.visible.to_numpy(), index=df.txId.values)
    #lookup from transaction ID to its visible flag (1 or 0)
    e = edges.assign(v1=edges.txId1.map(vis).fillna(0).to_numpy(), v2=edges.txId2.map(vis).fillna(0).to_numpy())
    #Adds two columns to every edge
    #v1: whether the source is visible
    #v2: whether the destination is visible. Missing IDs count as 0
    g = pd.DataFrame(index=df.txId.values)
    g["in_deg"] = e.groupby("txId2").size(); g["out_deg"] = e.groupby("txId1").size() 
    #incoming edges per transaction & outgoing edges per transaction
    g["in_vis"] = e.groupby("txId2").v1.sum(); g["out_vis"] = e.groupby("txId1").v2.sum() 
    #same counts for incoming and outgoing edges, but only from visible neighbours
    g = g.fillna(0)
    g["vis_frac"] = (g.in_vis + g.out_vis) / (g.in_deg + g.out_deg).replace(0, np.nan)
    g["in_vis_frac"] = g.in_vis / g.in_deg.replace(0, np.nan)
    g["out_vis_frac"] = g.out_vis / g.out_deg.replace(0, np.nan)
    #fraction of neighbours that are visible
    g.index = df.index
    return g


def hgb():
    return HistGradientBoostingClassifier(**HGB_PARAMS)


def lgbm():
    return lgb.LGBMClassifier(**LGBM_PARAMS)

`run_pipeline` takes a frame of fitting rows (all rows of the window, labelled or not) and rows to score, and returns the two D2 models' probabilities for the scored rows, plus the D2 LightGBM feature matrix the MLP uses.

In [ ]:
# builds every stage for one fitting window and returns D2 probabilities for the rows to score
def run_pipeline(df, fit_end, label):
    started = time.perf_counter()
    df = df.reset_index(drop=True)  # renumber rows 0..n-1 so array positions line up
    L = (df.role.eq("fit") & df.label.notna()).to_numpy() # labelled fitting rows
    P = df.role.eq("predict").to_numpy() # rows to score
    VIS = (L | P)
    df["visible"] = VIS.astype("int8")
    y = df.label[L].astype(int).to_numpy(); steps = df.time_step[L].to_numpy()
    w = 0.5 ** ((fit_end - steps) / HALF_LIFE)  # recency weight: halves every HALF_LIFE steps before fit_end

    # train on labelled fitting rows only, then predict the rows being scored
    def fit_predict(make, X):
        model = make().fit(X[L], y, sample_weight=w)
        return model.predict_proba(X[P])[:, 1], model

    # out-of-fold: each training row is scored by a model that never saw its label
    def oof(make, X):
        """Out-of-fold scores for labelled fitting rows (folds grouped by time step); full-fit scores for P."""
        s = np.full(len(X), np.nan); Xl = X[L]; idx = np.flatnonzero(L)
        for a, b in GroupKFold(N_FOLDS).split(Xl, y, steps):
            s[idx[b]] = make().fit(Xl.iloc[a], y[a], sample_weight=w[a]).predict_proba(Xl.iloc[b])[:, 1]
        # rows being scored get a score from a model trained on all fitting rows
        s[np.flatnonzero(P)] = fit_predict(make, X)[0]
        return s

    # stage A: the raw 165 features
    X0 = df[FEATURES]
    _, base = fit_predict(lgbm, X0)  # stage A LightGBM: feature ranking
    X1 = pd.concat([X0, structure(df)], axis=1) # stage B

    # the 30 features with the highest LightGBM gain
    top = pd.Series(base.booster_.feature_importance("gain"), index=FEATURES).sort_values(ascending=False).index[:TOP_FEATURES].tolist()
    NV = neighbour_agg(df, df[top], VIS, "nbv")  # visible-neighbour mean and max of the top 30 features
    diff = pd.DataFrame({f"own_minus_nbv_{c}": df[c] - NV[f"nbv_{c}_mean"] for c in top})  # own value minus its visible neighbours' mean
    X2 = pd.concat([X1, NV, diff], axis=1)  # stage C

    # outputs: the D2 predictions and the D2 LightGBM feature matrix
    preds, stage2 = {}, None
    # run the two stacking rounds once for HGB and once for LightGBM
    for make, column in [(hgb, "d2_hgb"), (lgbm, "d2_lightgbm")]:
        s1 = oof(make, X2)  # stage D1
        NS = neighbour_agg(df, pd.DataFrame({"s1": s1}), VIS & ~np.isnan(s1), "nbs", ("mean", "max", "min", "count"))  # round-1 summaries: mean, max, min and count of neighbours' scores
        s2 = oof(make, pd.concat([X2, NS], axis=1)) # stage D2
        NS2 = neighbour_agg(df, pd.DataFrame({"s2": s2}), VIS & ~np.isnan(s2), "nbs2")  # round-2 summaries: information from neighbours of neighbours
        Xs = pd.concat([X2, NS, NS2], axis=1)  # final feature matrix for this model
        preds[column] = fit_predict(make, Xs)[0]  # predict the rows being scored
        # keep the D2 LightGBM feature matrix (fitting and scored rows) for the MLP
        if column == "d2_lightgbm":
            keep = L | P
            stage2 = Xs[keep].assign(txId=df.txId[keep].to_numpy(), time_step=df.time_step[keep].to_numpy(),
                                     label=df.label[keep].to_numpy(), role=np.where(L[keep], "fit", "predict"))
    # keep only the scored rows and their identifiers
    out = df.loc[P, ["txId", "time_step", "label"]].reset_index(drop=True)
    for name, p in preds.items():
        out[name] = p
    # report run time
    print(f"{label}: D2 HGB and D2 LightGBM in {time.perf_counter() - started:.0f}s", flush=True)
    return out, stage2.reset_index(drop=True)  # stage2 is the input to the MLP

## 3. Run the pipeline inside each fitting window
**Everything is built inside each fitting window.** For the window 1–21, every feature, out-of-fold score and model uses fitting rows from steps 1–21 only; steps 22–35 are scored like test rows. Window 1–28 scores steps 29–35. Only labelled rows in the scored steps are scored. The final fit on steps 1–35 scores the test set.

In [ ]:
# group the checks by their fitting window (1–21 or 1–28)
windows = {}
for check in CHECKS:
    windows.setdefault(check["train_end"], []).append(check)

# validation predictions collected for each check
validation_frames = []
# one pipeline run per fitting window
for fit_end, window_checks in windows.items():
    # last step scored by any check in this window
    last = max(c["validation_end"] for c in window_checks)
    fit_rows = train[train.time_step <= fit_end].assign(role="fit")

    
    # labelled rows in the scored steps: rows to score
    scored = train[train.time_step.between(fit_end + 1, last) & train.label.notna()].assign(role="predict")
    # unknown-label rows in the scored steps: never visible, add no information
    others = train[train.time_step.between(fit_end + 1, last) & train.label.isna()].assign(role="other")
    # run every stage for this window
    out, stage2 = run_pipeline(pd.concat([fit_rows, scored, others], ignore_index=True), fit_end, f"window 1–{fit_end}")
    # save the D2 LightGBM feature matrix for the MLP step
    stage2.to_parquet(WORK / f"stage2_lightgbm_window_{fit_end}.parquet")
    # keep only the scored steps of each check
    for check in window_checks:
        validation_frames.append(out[out.time_step.between(check["validation_start"], check["validation_end"])].assign(check=check["name"]))


# all check predictions in one table
validation = pd.concat(validation_frames, ignore_index=True)
# labels back to integers for scoring
validation["label"] = validation.label.astype(int)

# final run: fit on steps 1–35 and score every test row
final_out, final_stage2 = run_pipeline(
    pd.concat([train.assign(role="fit"), test.drop(columns="index").assign(label=np.nan, role="predict")],
              ignore_index=True), 35, "final fit 1–35")
# feature matrix for the final MLP
final_stage2.to_parquet(WORK / "stage2_lightgbm_final.parquet")
final_out = final_out.merge(test[["index", "txId"]], on="txId", validate="one_to_one")

window 1–21: D2 HGB and D2 LightGBM in 111s


window 1–28: D2 HGB and D2 LightGBM in 125s


final fit 1–35: D2 HGB and D2 LightGBM in 139s


## 4. Stacked MLP
The MLP is trained on the **D2 LightGBM feature matrix** (the 165 features, structure and neighbour features, and both rounds of stacked neighbour scores). Features are mapped with a normal `QuantileTransformer` fitted on the labelled fitting rows; missing values become 0 with a missing-value flag per affected column. The network is 256–128 with BatchNorm, ReLU and 30% dropout, trained for 25 epochs (batch 512, AdamW, one-cycle learning rate) with the recency weights in the loss. Each batch also trains on an **FGSM** copy of itself (inputs nudged by ε = 0.03 in the direction that most increases the loss), so the network becomes less sensitive to small feature changes. Five seeds are averaged.

As above, the MLP is trained inside each fitting window (1–21 and 1–28) for the checks, and on all of steps 1–35 for the test predictions.

PyTorch runs in a **separate Python process**: running LightGBM and PyTorch in one process can deadlock on macOS. The next cell saves the MLP code as `stacked_mlp.py`; the cell after runs it.

In [ ]:
%%writefile stacked_mlp.py
"""Stacked MLP: reads the stage-2 feature matrices in work/ and writes 5-seed average probabilities."""
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import QuantileTransformer

# folder with the stage-2 matrices; predictions are written here
WORK = Path(sys.argv[1])
HALF_LIFE = 15 #same recency-weight half-life as the tree models
SEEDS = range(5) #five seeds averaged to reduce run-to-run variance
EPOCHS = 25
EPSILON = 0.03 #size of the adversarial (FGSM) nudge
META = ["txId", "time_step", "label", "role"]
torch.set_num_threads(8) # PyTorch threads on CPU


def mlp_predictions(stage2, fit_end):
    """Average of five seeds' probabilities for the rows to score in one stage-2 matrix."""
    X = stage2.drop(columns=META) # model inputs: the stage-2 features only
    L = stage2.role.eq("fit").to_numpy(); P = stage2.role.eq("predict").to_numpy() # masks for fitting rows (L) and rows to predict (P)
    flags = X.columns[X.isna().any()] # columns that contain at least one missing value
    # add a 0/1 missing-value flag for each of those columns
    X = pd.concat([X, X[flags].isna().astype("float32").add_suffix("_nan")], axis=1)
    qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", random_state=0).fit(X[L]) #each feature to a normal distribution
    Z = np.nan_to_num(qt.transform(X)).astype("float32")
    y = stage2.label[L].to_numpy().astype("float32")
    w = (0.5 ** ((fit_end - stage2.time_step[L].to_numpy()) / HALF_LIFE)).astype("float32") #recency weights, scaled to mean 1
    w /= w.mean()

    # convert to PyTorch tensors
    Xt, yt, wt = torch.tensor(Z[L]), torch.tensor(y), torch.tensor(w)
    lossf = nn.BCEWithLogitsLoss(reduction="none") #per-row loss

    # one prediction vector per seed
    seed_predictions = []
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)

        # 256 -> 128 -> 1 with BatchNorm and 30% dropout
        net = nn.Sequential(nn.Linear(Z.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(.3),
                            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(.3), nn.Linear(128, 1))

        # AdamW optimiser with weight decay
        opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-4)
        # one-cycle learning-rate schedule over all epochs
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, total_steps=EPOCHS * math.ceil(L.sum() / 512))

        # one pass over the fitting rows per epoch
        for _ in range(EPOCHS):
            net.train()
        
            for b in torch.randperm(len(Xt)).split(512): #shuffled mini-batches of 512 rows

                xb = Xt[b].clone().requires_grad_(True) #tracking gradients for the adversarial step
                loss = (lossf(net(xb).squeeze(1), yt[b]) * wt[b]).mean() # recency-weighted loss

                # gradient of the loss with respect to the inputs
                grad, = torch.autograd.grad(loss, xb, retain_graph=True)
                x_adv = (xb + EPSILON * grad.sign()).detach()      # FGSM worst-case nudge

                # the same loss on the adversarial copy of the batch
                loss_adv = (lossf(net(x_adv).squeeze(1), yt[b]) * wt[b]).mean()
                # train on the average of the clean and adversarial losses
                opt.zero_grad(); (0.5 * loss + 0.5 * loss_adv).backward(); opt.step(); sched.step()

        # switch off dropout for prediction
        net.eval()
        with torch.no_grad():
            # this seed's probabilities for the rows to predict
            seed_predictions.append(torch.sigmoid(net(torch.tensor(Z[P])).squeeze(1)).numpy().astype(np.float64))
            
    #identifiers for the predicted rows
    out = stage2.loc[P, ["txId", "time_step", "label"]].reset_index(drop=True)
    # average the five seeds
    out["mlp"] = np.mean(seed_predictions, axis=0)
    return out


#run all three windows
if __name__ == "__main__":
    print("PyTorch:", torch.__version__, flush=True)
    for name, fit_end in [("window_21", 21), ("window_28", 28), ("final", 35)]:
        out = mlp_predictions(pd.read_parquet(WORK / f"stage2_lightgbm_{name}.parquet"), fit_end)

        # save predictions for the notebook to read (not submitted to github repo)
        out.to_csv(WORK / f"mlp_predictions_{name}.csv", index=False)
        print(f"{name}: MLP done", flush=True)

Overwriting stacked_mlp.py


In [ ]:
# run the MLP in a separate Python process!!! (avoids the LightGBM/PyTorch clash as this happened in my environment)
started = time.perf_counter()
run = subprocess.run([sys.executable, "stacked_mlp.py", str(WORK)], capture_output=True, text=True)
print(run.stdout)

if run.returncode:
    raise RuntimeError(run.stderr)
print(f"MLP step finished in {time.perf_counter() - started:.0f}s")


KEYS = ["check", "txId", "time_step", "label"]

# MLP predictions split by check, as for the trees
mlp_frames = []
for fit_end, window_checks in windows.items():
    # predictions of this window
    out = pd.read_csv(WORK / f"mlp_predictions_window_{fit_end}.csv")
    for check in window_checks:
        mlp_frames.append(out[out.time_step.between(check["validation_start"], check["validation_end"])].assign(check=check["name"]))


# all MLP check predictions in one table
mlp_validation = pd.concat(mlp_frames, ignore_index=True)
mlp_validation["label"] = mlp_validation.label.astype(int)

# attach MLP predictions to the tree predictions by check and transaction
validation = validation.merge(mlp_validation, on=KEYS, validate="one_to_one")
assert len(validation) == len(mlp_validation)
# every test row with all three model predictions
test_predictions = final_out.merge(pd.read_csv(WORK / "mlp_predictions_final.csv")[["txId", "mlp"]],
                                   on="txId", validate="one_to_one")
assert len(test_predictions) == len(test)

PyTorch: 2.14.1
window_21: MLP done
window_28: MLP done
final: MLP done

MLP step finished in 80s


## 5. Validation of the three models

In [9]:
def score_checks(predictions, column="illicit_probability"):
    """ROC-AUC and average precision for each check."""
    rows = []
    for check in CHECKS:
        group = predictions.loc[predictions.check.eq(check["name"])]
        rows.append({"check": check["name"],
                     "fit_steps": f'{check["train_start"]}–{check["train_end"]}',
                     "validation_steps": f'{check["validation_start"]}–{check["validation_end"]}',
                     "validation_rows": len(group), "validation_illicit": int(group.label.sum()),
                     "validation_roc_auc": roc_auc_score(group.label, group[column]),
                     "average_precision": average_precision_score(group.label, group[column])})
    return pd.DataFrame(rows)


# one row per model: ROC-AUC and average precision on each check
display(pd.DataFrame([{"model": name, **score_checks(validation, column).set_index("check").validation_roc_auc.to_dict()}
                      for name, column in [("D2 LightGBM", "d2_lightgbm"), ("D2 HGB", "d2_hgb"),
                                           ("Stacked MLP (5 seeds)", "mlp")]]).round(5))

,model,steps_22_28,steps_29_35,14_step_check
0,D2 LightGBM,0.93420,0.99876,0.86396
1,D2 HGB,0.93230,0.99808,0.86086
2,Stacked MLP (5 seeds),0.97359,0.99513,0.94743


## 6. Blend ratios
Each model's probabilities are converted to ranks within the scored rows (for the checks, within each check), then combined as LGBM : HGB : MLP. Every distinct ratio with integer weights 0–3 is scored on the three checks and ranked by the mean of the two 7-step checks; ratios are reduced to lowest terms.

In [10]:
# the three models being blended, in the order used by each ratio
COMPONENTS = ["d2_lightgbm", "d2_hgb", "mlp"]
# every ratio with integer weights 0–3, reduced to lowest terms (49 in total)
RATIOS = sorted({tuple(int(v) for v in np.array(r) // math.gcd(*r))
                 for r in itertools.product(range(4), repeat=3) if any(r)})


# within-set ranks scaled to [0, 1], then the weighted sum
def rank_blend(frame, ratio):
    return sum(weight * (rankdata(frame[c]) / len(frame)) for weight, c in zip(ratio, COMPONENTS))


# AUC of every ratio on each check
rows = []
for ratio in RATIOS:
    # ratio label, e.g. 3:1:1
    row = {"LGBM:HGB:MLP": ":".join(map(str, ratio))}
    # score this ratio on each check
    for check in CHECKS:
        part = validation.loc[validation.check.eq(check["name"])]
        row[check["name"]] = roc_auc_score(part.label, rank_blend(part, ratio))
    rows.append(row)
# table of all ratios
sweep = pd.DataFrame(rows)
# selection score: mean of the two 7-step checks
sweep["Mean of 7-step checks"] = sweep[SELECTION_CHECKS].mean(axis=1)
# best selection score first
sweep = sweep.sort_values("Mean of 7-step checks", ascending=False).reset_index(drop=True)
# rank 1 = best selection score
sweep.insert(0, "Rank", np.arange(1, len(sweep) + 1))
# readable column names for display and the report
sweep = sweep.rename(columns={"steps_22_28": "Steps 22–28", "steps_29_35": "Steps 29–35",
                              "14_step_check": "14-step check (22–35)"})
# show every row
with pd.option_context("display.max_rows", 60):
    display(sweep.round(5))

,Rank,LGBM:HGB:MLP,Steps 22–28,Steps 29–35,14-step check (22–35),Mean of 7-step checks
0,1,0:0:1,0.97359,0.99513,0.94743,0.98436
1,2,1:0:3,0.97079,0.99742,0.93808,0.98410
2,3,0:1:3,0.97063,0.99736,0.93758,0.98399
3,4,1:0:2,0.96907,0.99780,0.93300,0.98344
4,5,0:1:2,0.96888,0.99769,0.93241,0.98329
5,6,1:1:3,0.96735,0.99802,0.92814,0.98268
6,7,2:0:3,0.96726,0.99804,0.92822,0.98265
7,8,0:2:3,0.96705,0.99791,0.92756,0.98248
8,9,2:1:3,0.96406,0.99832,0.91985,0.98119
9,10,1:0:1,0.96400,0.99834,0.91974,0.98117


The stacked MLP alone and the blend ratios compared:

In [ ]:
# ratios discussed in the report
COMPARED = ["0:0:1", "1:0:3", "1:1:3", "1:1:1", "2:2:1", "1:1:0", "3:3:1", "3:2:1", "3:1:1"]

# columns shown in the comparison
COLUMNS = ["Rank", "LGBM:HGB:MLP", "Steps 22–28", "Steps 29–35", "14-step check (22–35)", "Mean of 7-step checks"]
# the comparison table, taken from the sweep above
display(sweep.loc[sweep["LGBM:HGB:MLP"].isin(COMPARED), COLUMNS].round(5))

,Rank,LGBM:HGB:MLP,Steps 22–28,Steps 29–35,14-step check (22–35),Mean of 7-step checks
0,1,0:0:1,0.97359,0.99513,0.94743,0.98436
1,2,1:0:3,0.97079,0.99742,0.93808,0.98410
5,6,1:1:3,0.96735,0.99802,0.92814,0.98268
24,25,1:1:1,0.95606,0.99860,0.90151,0.97733
34,35,3:1:1,0.94777,0.99881,0.88502,0.97329
35,36,2:2:1,0.94769,0.99872,0.88479,0.97321
37,38,3:2:1,0.94556,0.99878,0.88089,0.97217
39,40,3:3:1,0.94390,0.99873,0.87789,0.97132
43,44,1:1:0,0.93444,0.99853,0.86293,0.96648


## 7. Final submissions
Each submission ranks the three models' test probabilities from the final fit on steps 1–35, applies the ratio and divides by the maximum so every target lies in [0, 1].

In [ ]:
sample = pd.read_csv(DATA / "sample_submission.csv")
assert set(test_predictions["index"]) == set(sample["index"]) #all test rows have a prediction


for ratio in FINAL_RATIOS:
    blend = rank_blend(test_predictions, ratio)
    submission = sample.copy()
    # scale to [0, 1] by dividing by the maximum
    submission["target"] = submission["index"].map(pd.Series(blend / blend.max(), index=test_predictions["index"]))
    assert np.isfinite(submission.target).all() and submission.target.between(0, 1).all()

    # output file in the submission folder 
    path = SUBMISSIONS / f"graph_blend_lgbm_hgb_mlp_{''.join(map(str, ratio))}.csv"
    submission.to_csv(path, index=False)
    saved = pd.read_csv(path)
    
    # format check: index,target
    assert list(saved.columns) == ["index", "target"]
    assert saved["index"].equals(sample["index"]) and np.allclose(saved.target, submission.target, atol=1e-12)
    print(f"Saved {len(saved):,} probabilities to submission/{path.name}")

Saved 15,329 probabilities to submission/graph_blend_lgbm_hgb_mlp_311.csv
Saved 15,329 probabilities to submission/graph_blend_lgbm_hgb_mlp_111.csv


## Findings
1. On the training checks the stacked MLP is the strongest single graph model: 0.97359 / 0.99513 / 0.94743 on steps 22–28 / 29–35 / 14-step, against 0.93420 / 0.99876 / 0.86396 for D2 LightGBM and 0.93230 / 0.99808 / 0.86086 for D2 HGB.
2. Blends rank higher the more MLP weight they carry: the MLP alone ranks 1st (mean 7-step AUC 0.98436), 1:0:3 ranks 2nd (0.98410), 1:1:3 6th (0.98268) and 1:1:1 25th (0.97733). The tree-only blends rank last.
3. Steps 29–35 move the other way: more tree weight scores slightly higher there (1:1:1 0.99860, MLP alone 0.99513), but every ratio lies between 0.995 and 0.999, so steps 22–28 and the 14-step check drive the ranking.
4. The differences at the top of the table are small (0.98436 vs 0.98410 for the first two rows).